# 🏦 Pandas for auditors — Exercises Level 3: Advanced

**Context**: You are an auditor in the **Compliance / AML/CFT** team (Anti-Money
Laundering and Counter-Terrorist Financing) of a private bank.

The IT department provided you with two files:
- **`flows`**: the financial flows for the period (transactions)
- **`clients`**: the client reference table with their KYC risk profile

Your mission: enrich the data, identify atypical behaviors,
and produce a **risk score** per client.

**Skills covered**: `merge`, time-based analysis (`.dt`), pattern detection (structuring,
cash, risk countries), advanced `groupby`, combining criteria, export.

> ℹ️ The data and the alert criteria are **fictional and simplified** for educational purposes.

## 0. Data generation — run this first

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
np.random.seed(303)

# ── Client reference table ────────────────────────────────────────────────────
n_clients = 60
client_ids = [f'CLI{str(i).zfill(4)}' for i in range(1, n_clients + 1)]

clients = pd.DataFrame({
    'client_id':       client_ids,
    'segment':         np.random.choice(['Individual', 'Corporate', 'Private Banking'],
                                        n_clients, p=[0.45, 0.35, 0.20]),
    'residence_country':  np.random.choice(['FR', 'LU', 'CH', 'MC', 'BE', 'DE'],
                                        n_clients, p=[0.50, 0.15, 0.12, 0.08, 0.10, 0.05]),
    'kyc_risk_level': np.random.choice(['Low', 'Standard', 'High'],
                                          n_clients, p=[0.40, 0.45, 0.15]),
    'relationship_start_date': pd.to_datetime('2015-01-01') + pd.to_timedelta(
        np.random.randint(0, 3285, n_clients), unit='D'
    ),
})

# ── Transactions ──────────────────────────────────────────────────────────────
n = 600

counterparty_country = np.random.choice(
    ['FR', 'DE', 'BE', 'LU', 'CH', 'US', 'GB', 'AE', 'PA', 'CY', 'MT', 'SG'],
    n, p=[0.20, 0.12, 0.10, 0.10, 0.08, 0.08, 0.07, 0.06, 0.05, 0.05, 0.05, 0.04]
)

op_types = np.random.choice(
    ['Incoming Transfer', 'Outgoing Transfer', 'Cash Deposit', 'Cash Withdrawal',
     'Check', 'Direct Debit'],
    n, p=[0.28, 0.28, 0.10, 0.10, 0.12, 0.12]
)

dates = pd.to_datetime('2024-01-01') + pd.to_timedelta(
    np.random.randint(0, 366, n), unit='D'
)

base_amounts = np.round(np.random.lognormal(mean=7.0, sigma=1.3, size=n), 2)

flows = pd.DataFrame({
    'flow_id':          [f'FL{str(i).zfill(6)}' for i in range(1, n + 1)],
    'client_id':        np.random.choice(client_ids, n),
    'date':             dates,
    'operation_type':   op_types,
    'amount':           base_amounts,
    'currency':         np.random.choice(['EUR', 'USD', 'CHF', 'GBP'],
                                         n, p=[0.78, 0.10, 0.08, 0.04]),
    'counterparty_country': counterparty_country,
    'channel':          np.random.choice(['SWIFT', 'SEPA', 'Internal', 'Counter'],
                                         n, p=[0.25, 0.40, 0.20, 0.15]),
})

# ── Intentional traps ─────────────────────────────────────────────────────────
# Structuring: amounts just under 10,000
idx_struct = np.random.choice(flows.index, 12, replace=False)
flows.loc[idx_struct, 'amount'] = np.random.choice([9500, 9750, 9800, 9900, 9950, 9990], 12)
flows.loc[idx_struct, 'operation_type'] = 'Cash Deposit'

# Large cash amounts
idx_cash = np.random.choice(flows.index, 8, replace=False)
flows.loc[idx_cash, 'amount'] = np.random.choice([15000, 20000, 25000, 30000], 8)
flows.loc[idx_cash, 'operation_type'] = np.random.choice(['Cash Deposit', 'Cash Withdrawal'], 8)

# Risk countries (simplified fictional list)
risk_country = ['AE', 'PA', 'CY']
idx_risk = np.random.choice(flows.index, 15, replace=False)
flows.loc[idx_risk, 'counterparty_country'] = np.random.choice(risk_country, 15)
flows.loc[idx_risk, 'amount'] = np.round(np.random.lognormal(mean=9.0, sigma=0.8, size=15), 2)

# Missing values
flows.loc[np.random.choice(flows.index, 15, replace=False), 'counterparty_country'] = np.nan

flows = flows.sample(frac=1, random_state=5).reset_index(drop=True)

print('Flows ready:', flows.shape)
print('Clients ready:', clients.shape)

**Table descriptions**

**Table `flows`**

| Column | Description |
|---|---|
| `flow_id` | flow identifier |
| `client_id` | client identifier |
| `date` | flow date |
| `operation_type` | Incoming/Outgoing Transfer, Cash Deposit/Withdrawal, Check, Direct Debit |
| `amount` | amount in currency |
| `currency` | EUR / USD / CHF / GBP |
| `counterparty_country` | ISO country code of the counterparty |
| `channel` | SWIFT / SEPA / Internal / Counter |

**Table `clients`**

| Column | Description |
|---|---|
| `client_id` | client identifier |
| `segment` | Individual / Corporate / Private Banking |
| `residence_country` | country of residence |
| `kyc_risk_level` | Low / Standard / High |
| `relationship_start_date` | relationship start date |

**Risk countries (simplified fictional list)**: `AE`, `PA`, `CY`

---
## Exercise 1 — Enrichment with `merge`

Before analyzing the flows, you need to **bring the client information** into the flows table.

**Questions:**
1. Merge `flows` and `clients` on `client_id` (left join — `how='left'`).
   Call the result `flows_enriched`. Check the number of columns obtained.
2. Are there any flows for which the client is **not found** in the reference table?
   (check the `NaN` values in the `kyc_risk_level` column after the merge)
3. What is the **total amount** of flows by **KYC risk level**?
   Sort from highest to lowest.
4. How many flows involve **`Private Banking`** clients residing **outside France**?

In [ ]:
# 1. Merge flows + clients
# Your code here


In [ ]:
# 2. Flows with client not found
# Your code here


In [ ]:
# 3. Total amount by KYC risk level
# Your code here


In [ ]:
# 4. Private Banking flows outside France
# Your code here


---
## Exercise 2 — Time-based analysis

In AML/CFT, the **timing** of transactions can be an alert signal.

**Questions:**
1. Extract into `flows_enriched` the columns **`month`**, **`weekday`** (0=Monday … 6=Sunday)
   and **`day_name`** from the `date` column.
2. Compute the **total amount per month**. Which month is the most active by volume?
3. Isolate the flows made on the **weekend** (Saturday or Sunday). How many are there
   in number and as a percentage of the total?
4. Among the weekend flows, which are **`Cash Deposit`** or **`Cash Withdrawal`**?
   Display the columns `date`, `day_name`, `client_id`, `operation_type`, `amount`.

In [ ]:
# 1. Extract month, weekday, day_name
# Your code here


In [ ]:
# 2. Total amount per month
# Your code here


In [ ]:
# 3. Weekend flows (count and %)
# Your code here


In [ ]:
# 4. Cash on weekends
# Your code here


---
## Exercise 3 — Detecting *structuring* (smurfing)

**Structuring** consists of splitting flows into amounts just below
the reporting threshold of **€10,000** to avoid controls.

**Questions:**
1. Identify the flows whose amount is **between €9,000 and €9,999.99** (inclusive).
   How many are there?
2. Among these suspicious flows, what is the most frequent operation type?
3. For each **client** with at least **2 flows** in the 9,000–9,999 range,
   compute the number of these flows and the total amount.
   This client deserves special attention — what is their `kyc_risk_level`?
4. Create a boolean column **`structuring_alert`** in `flows_enriched`:
   `True` if the amount is between 9,000 and 9,999 **AND** the operation is Cash.

In [ ]:
# 1. Flows in the 9,000 – 9,999 range
# Your code here


In [ ]:
# 2. Most frequent operation type among the suspects
# Your code here


In [ ]:
# 3. Clients with >= 2 flows in the 9,000–9,999 range
# Your code here


In [ ]:
# 4. structuring_alert column
# Your code here


---
## Exercise 4 — Cash flow analysis

Cash operations are particularly monitored in AML/CFT.

**Questions:**
1. Isolate all flows of type **`Cash Deposit`** or **`Cash Withdrawal`**.
2. For each **client**, compute the **total** of cash amounts (all operations combined).
   Identify the 10 clients with the highest total. What is their `kyc_risk_level`?
3. Create a column **`cash_alert`** in `flows_enriched`:
   `True` if the flow is cash **AND** the amount exceeds **€10,000**.
4. How many distinct clients have at least **one** cash alert?

In [ ]:
# 1. Cash flows
# Your code here


In [ ]:
# 2. Top 10 clients by cash total
# Your code here


In [ ]:
# 3. cash_alert column
# Your code here


In [ ]:
# 4. Number of clients with a cash alert
# Your code here


---
## Exercise 5 — Flows to risk countries

The fictional list of **risk countries** for this exercise is: `['AE', 'PA', 'CY']`.

**Questions:**
1. Create a column **`risk_country`** in `flows_enriched`: `True` if `counterparty_country`
   is in the list of risk countries.
2. What is the **total amount** of flows to risk countries, by **operation type**?
3. Identify the clients with **at least 3 flows** to risk countries.
   Display their `client_id`, `segment`, `kyc_risk_level`, number of flows and total amount.
4. Among the flows to risk countries, which **`High` KYC risk clients** are involved?

In [ ]:
# 1. risk_country column
risk_country_list = ['AE', 'PA', 'CY']
# Your code here


In [ ]:
# 2. Total amount to risk countries by operation type
# Your code here


In [ ]:
# 3. Clients with >= 3 flows to risk countries
# Your code here


In [ ]:
# 4. High KYC clients involved in flows to risk countries
# Your code here


---
## Exercise 6 — Multi-criteria risk scoring (synthesis)

You need to produce a **risk dashboard** per client, combining all the alerts.

**Build a table with, for each client:**
- `total_flows`: total number of flows
- `total_amount`: total amount
- `structuring_alerts_count`: number of flows with `structuring_alert == True`
- `cash_alerts_count`: number of flows with `cash_alert == True`
- `risk_country_flows`: number of flows to risk countries
- `risk_score`: sum of the three alert counters
- `kyc_risk_level`: brought in from the `clients` table
- `segment`: brought in from the `clients` table

Sort by `risk_score` descending and display the **10 highest-risk clients**.

Export this table to a file named **`aml_risk_report.xlsx`**.

In [ ]:
# Multi-criteria risk scoring
# Your code here


In [ ]:
# Excel export
# Your code here
